In [31]:
import pandas as pd
import numpy as np
import seaborn as sns

# Quiz Questions

Use this dataset to do the following tasks. Note that you'll need to do some cleaning to make things work, I suggest checking the questions for what you need first, as the last prompt requires some cleaning that the other ones don't.
<ul>
<li> Are colleges that require the JEE MAIN exam more expensive than the other colleges? Support your conclusion. </li>
<li> Create a model to predict the One_Year_Fee. </li>
<li> Add the Place and Approved_By and use them as features in the model. </li>
</ul>

In [32]:
df_college = pd.read_csv("data/college_data.csv")
df_college.head()

,Unnamed: 0,College_Name,One_Year_Fee,Enterance_Exam,Rating,Review_Counts,Approved_By,Place,More_Details
0,0,BITS Pilani (Pilani Campus),"₹ 541,575",BITSAT,8.8,(501 Reviews),['UGC'],"Pilani,Rajasthan",https://collegedunia.com/university/25800-bits...
1,1,Institute of Chemical Technology - [ICT],"₹ 89,100",JEE MAIN,8.7,(45 Reviews),"['UGC, NBA, MHRD']","Mumbai,Maharashtra",https://collegedunia.com/university/25705-inst...
2,2,Woxsen University,"₹ 372,500",VITEEE,8.2,(138 Reviews),"['AICTE, COA, NBA, AACSB, AMBA']","Hyderabad,Telangana",https://collegedunia.com/university/60661-woxs...
3,3,Vellore Institute of Technology - [VIT Univers...,"₹ 198,000",VITEEE,8.4,(1660 Reviews),"['AICTE, UGC']","Vellore,TamilNadu",https://collegedunia.com/university/25914-vell...
4,4,East Point Group of Institutions - [EPGI],"₹ 250,000",COMEDK UGET,7.2,(93 Reviews),"['AICTE, NBA']","Bangalore,Karnataka",https://collegedunia.com/college/13428-east-po...


### Are Colleges Requiring the JEE MAIN Exam More Expensive? (4pts)

Some colleges require the JEE MAIN exam for admission, while others do not. Does it appear that there is a meaningful difference in the One_Year_Fee between these two groups? Show how you determined your answer. 

In [33]:
print(df_college.columns.tolist())

['Unnamed: 0', 'College_Name', 'One_Year_Fee', 'Enterance_Exam', 'Rating', 'Review_Counts', 'Approved_By', 'Place', 'More_Details']


In [34]:
# Question 1

# Reload the original dataset
df_college = pd.read_csv("data/college_data.csv")

# Clean column names
df_college.columns = df_college.columns.str.strip()

# Convert fees into numbers
df_college["One_Year_Fee"] = pd.to_numeric(
    df_college["One_Year_Fee"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

# Identify JEE MAIN colleges
df_college["JEE_MAIN"] = (
    df_college["Enterance_Exam"]
    .fillna("")
    .astype(str)
    .str.contains("JEE MAIN", case=False, regex=False)
)

# Compare the groups
comparison = df_college.groupby("JEE_MAIN")["One_Year_Fee"].agg(
    ["count", "mean", "median"]
)

print(comparison)

          count           mean    median
JEE_MAIN                                
False       264  142980.659091  100000.0
True        263  166465.015209  126000.0


Based on the results, colleges that require JEE MAIN seem to be more expensive. Their average yearly fee is ₹166,465, compared to ₹142,981 for other colleges. The median is also higher, at ₹126,000 compared to ₹100,000. This suggests that JEE MAIN colleges generally cost more, but we would need a statistical test to know if the difference is significant.

### Create a Regression Model (4pts)

Make a model to predict the One_Year_Fee, using the variables:
<ul>
	<li>Enterance_Exam</li>
	<li>Rating</li>
	<li>Review_Counts</li>
</ul>

In [35]:

# Question 2: Predict One_Year_Fee

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare the data
df_model = df_college.copy()

# Clean the review counts
df_model["Review_Counts"] = pd.to_numeric(
    df_model["Review_Counts"].astype(str)
    .str.replace(r"[^0-9]", "", regex=True),
    errors="coerce"
)

# Make sure rating and fees are numeric
df_model["Rating"] = pd.to_numeric(
    df_model["Rating"], errors="coerce"
)

# Convert entrance exams into numeric columns
df_model["Enterance_Exam"] = df_model["Enterance_Exam"].fillna("Unknown")

df_model = pd.get_dummies(
    df_model,
    columns=["Enterance_Exam"],
    dtype=int
)

# Select model features
features = [
    col for col in df_model.columns
    if col.startswith("Enterance_Exam_")
] + ["Rating", "Review_Counts"]

# Remove rows with missing values
df_model = df_model.dropna(
    subset=["One_Year_Fee"] + features
)

X = df_model[features]
y = df_model["One_Year_Fee"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate predictions
predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("R-squared:", r2_score(y_test, predictions))
print("Rows used:", len(df_model))


MAE: 82282.31196417905
R-squared: 0.11077319065626812
Rows used: 527


I used Entrance_Exam, Rating, and Review_Counts to predict the one-year college fees. The model had an MAE of about ₹82,282 and an R² of 0.111. This means it only explains about 11.1% of the differences in college fees, so the model is not very accurate at making predictions.

### Add the Place and Approved_By to the Feature Set* (2pts)

Now, add the Place and Approved_By to the feature set - these each have one column representing multiple values, which is bad. During processing, if one of those values occurs less than M times (use 10 as a value), then we will combine it with the other "other" values. This applies to the individual values in the Approved_By column, so if there are 5 colleges approved by "AICTE" and 3 colleges approved by "UGC", then we would combine those into a single "other" value.

<b>This will require some unique data processing.</b>

This is not worth a lot of points, so finish other stuff first if this is difficult. 

In [36]:

# Question 3: Add Place and Approved_By

import pandas as pd
import ast

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MultiLabelBinarizer

# Load the original dataset
df_q3 = pd.read_csv("data/college_data.csv")

# Clean column names
df_q3.columns = df_q3.columns.str.strip()

# Clean One_Year_Fee
df_q3["One_Year_Fee"] = pd.to_numeric(
    df_q3["One_Year_Fee"].astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

# Clean Review_Counts
df_q3["Review_Counts"] = pd.to_numeric(
    df_q3["Review_Counts"].astype(str)
    .str.extract(r"(\d[\d,]*)")[0]
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# Clean Rating
df_q3["Rating"] = pd.to_numeric(
    df_q3["Rating"], errors="coerce"
)

# Clean Entrance Exam
df_q3["Enterance_Exam"] = (
    df_q3["Enterance_Exam"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# Clean Place
df_q3["Place"] = (
    df_q3["Place"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# Group places appearing fewer than 10 times
place_counts = df_q3["Place"].value_counts()

df_q3["Place"] = df_q3["Place"].where(
    df_q3["Place"].map(place_counts) >= 10,
    "other"
)

# Convert Approved_By strings into lists
def parse_approvals(value):
    if pd.isna(value):
        return []

    try:
        values = ast.literal_eval(str(value))
        if isinstance(values, list):
            return [str(x).strip() for x in values]
    except (ValueError, SyntaxError):
        pass

    return []

df_q3["Approval_List"] = (
    df_q3["Approved_By"].apply(parse_approvals)
)

# Count each individual approval
approval_counts = (
    df_q3["Approval_List"]
    .explode()
    .value_counts()
)

# Group approvals appearing fewer than 10 times
df_q3["Approval_List"] = df_q3["Approval_List"].apply(
    lambda values: list(set(
        x if approval_counts.get(x, 0) >= 10 else "other"
        for x in values
    ))
)

# Create numeric columns for approvals
mlb = MultiLabelBinarizer()

approval_data = pd.DataFrame(
    mlb.fit_transform(df_q3["Approval_List"]),
    columns=["Approval_" + x for x in mlb.classes_],
    index=df_q3.index
)

# Convert Entrance Exam and Place into numeric columns
df_q3 = pd.get_dummies(
    df_q3,
    columns=["Enterance_Exam", "Place"],
    dtype=int
)

# Add approval columns
df_q3 = pd.concat(
    [df_q3, approval_data],
    axis=1
)

# Select model features
# IMPORTANT: Exclude Approval_List because it contains lists
features_q3 = [
    col for col in df_q3.columns
    if col.startswith(("Enterance_Exam_", "Place_"))
    or (
        col.startswith("Approval_")
        and col != "Approval_List"
    )
] + ["Rating", "Review_Counts"]

# Prepare X and y
X = df_q3[features_q3].copy()
X = X.apply(pd.to_numeric, errors="coerce")

y = df_q3["One_Year_Fee"]

# Remove rows with missing values
valid_rows = X.notna().all(axis=1) & y.notna()

X = X.loc[valid_rows]
y = y.loc[valid_rows]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train the regression model
model_q3 = LinearRegression()
model_q3.fit(X_train, y_train)

# Make predictions
predictions_q3 = model_q3.predict(X_test)

# Display results
print("Question 3 Results")
print("MAE:", mean_absolute_error(y_test, predictions_q3))
print("R-squared:", r2_score(y_test, predictions_q3))
print("Rows used:", len(X))
print("Features used:", len(features_q3))


Question 3 Results
MAE: 90266.73704404931
R-squared: -0.007990184627210795
Rows used: 527
Features used: 79


After adding Place and Approved_By, the model performed worse. The MAE went up from ₹82,282 to ₹90,267, and the R² dropped from 0.111 to -0.008. This means adding these two variables did not help the model make better predictions. It may have made the model more complicated without improving the results.